In [21]:
using LowLevelFEM, LinearAlgebra
using IterativeSolvers, Preconditioners

In [22]:
openGeometry("I-beam.geo")

In [23]:
openPreProcessor()

In [24]:
mat = Material("beam")
P = Material("remote")
U = Field([mat, P], type=:VectorField, dim=3, fieldName=:u)
Φ = Field([mat, P], type=:VectorField, dim=3, fieldName=:φ)

Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 281269, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))

In [25]:
mpc_U = MultiPointConstraint(master="remote", slave="right", field=U)
mpc_Φ = MultiPointConstraint(master="remote", slave="right", field=Φ)

MultiPointConstraint("remote", "right", Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 281269, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), Dict{Symbol, Bool}())

In [26]:
bc = BoundaryCondition("left", field=U, ux=0, uy=0)
bcz = BoundaryCondition("left", field=U, uz=0)
bcφ = BoundaryCondition("remote", field=Φ, φz=0.001)

BoundaryCondition("remote", Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 281269, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), Dict{Symbol, Union{Function, Number, ScalarField}}(:φz => 0.001))

In [27]:
Ku = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="beam")
Ku[:,:]

843807×843807 SparseArrays.SparseMatrixCSC{Float64, Int64} with 56460533 stored entries:
⎡⡿⣯⡭⢿⡿⠿⠿⠿⠿⠿⠿⠟⠛⠿⠶⠾⠷⠶⠷⠿⠿⢽⣿⡿⠿⠿⠿⠿⠿⠿⠟⠿⠿⠶⠾⠷⠶⠷⠿⠾⎤
⎢⣧⣏⣻⣾⡄⠀⠀⠀⠤⠀⠀⢀⣀⡀⠀⠀⠀⠀⠀⠀⢘⣯⣥⡀⠀⠤⠀⠠⠀⠀⣀⣀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡏⠀⠉⠻⣦⡁⠀⢸⠀⠀⠉⠀⠉⠀⠀⠀⠀⠀⠀⠀⠀⠉⢁⡴⠋⠀⠀⠈⠁⠀⠈⠉⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡇⠀⠀⠁⠈⠻⣦⣸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠀⠘⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡇⠀⠃⠒⠒⠒⠚⠻⣦⡀⡆⠐⠒⠒⠒⠒⠒⠂⠀⠀⠀⠀⠒⠒⠒⠒⠛⠲⢤⡀⠀⠐⠒⠒⠒⠒⠂⠀⠀⎥
⎢⣿⠇⠀⢀⡄⠀⠀⠀⠠⠬⠿⣧⡀⢠⠀⠀⠀⠀⠤⠤⠤⠀⠀⠀⠀⠀⠀⠰⠾⠭⢅⣠⡤⠀⠀⠀⠀⠤⠤⠤⎥
⎢⣿⡄⠀⠸⡄⠀⠀⠀⢰⠀⠀⣈⣻⣾⡀⢀⡀⠀⡄⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣛⣷⣤⠀⢰⡄⠀⠆⣀⠀⎥
⎢⣸⡇⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⢈⠻⣦⡀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡴⠋⠀⠀⠀⠀⠀⎥
⎢⢹⡇⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠀⠈⠀⠈⠻⣦⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⠀⠻⣦⡀⠀⠀⎥
⎢⣽⡇⠀⠀⠀⠀⠀⠀⠈⠀⠀⡇⠀⠉⠉⠉⠉⠉⠻⣦⣸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠉⠉⠉⠙⢲⣤⎥
⎢⣟⣇⡶⣴⠀⠀⠀⠀⠀⠀⠀⠃⠀⠐⠀⠀⠀⠀⠒⠚⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠚⠛⠓⎥
⎢⣿⡿⠁⠻⠇⢀⡄⠀⢠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⢠⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡇⠀⡄⡴⠋⣀⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣈⠻⣦⡙⢦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡇⠀⡀⠀⠀⠈⠳⣼⠀⢀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣌⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡇⠀⠀⠆⠀⠀⠀⠘⣆⡞⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡅⠀⢸⡀⠀⠀⠀⠀⠈⠁⣱⢿⣼⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢻⡇⠀⠀⠃⠀⠀⠀⢰⠀⠀⠋⠀⠛⢀⡴⠂⠀⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⢦⡀⠀⠀⠀⎥
⎢⢾⡇⠀⠀⠀⠀⠀⠀⢸⠀⠀⠀⠐⠶⠋⠀⣤⡀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠠⣌⠻⣦⡙⠆⠀⠀⎥
⎢⢼⡇⠀⠀⠀⠀⠀⠀⠸⠀⠀⡄⠠⠄⠀⠀⠈⠻⣇⠀⣠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⠌⠻⣦⡀⠀⎥
⎣⣻⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⡇⠀⠘⠀⠀⠀⠀⠘⣶⢿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [28]:
k = 1e4
Kc = ∫(U ⋅ k ⋅ U, Γ="left")
#Ku += Kc
Kc[:,:]

843807×843807 SparseArrays.SparseMatrixCSC{Float64, Int64} with 10470 stored entries:
⎡⠅⠅⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [29]:
R = rigidRotationMap(mpc_U, mpc_Φ);

In [30]:
K = SystemMatrix([Ku Ku * R; R' * Ku R' * Ku * R])
K[:,:]

1687614×1687614 SparseArrays.SparseMatrixCSC{Float64, Int64} with 56617245 stored entries:
⎡⣿⣿⡟⠛⡛⠛⠛⠛⠛⠛⢻⣟⢛⢛⠛⠛⠛⠛⠛⠛⠛⠀⠀⠀⠀⠀⠘⠀⠘⠀⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⣿⠉⠿⣧⡇⠈⠉⠀⠀⠀⠈⠩⠾⣄⠈⠉⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⡉⠉⣻⣾⣈⠉⠉⣁⡀⠈⠉⢉⣷⣆⣈⠉⢉⣀⡀⠀⠀⠀⠀⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠃⠀⡆⠘⠻⣦⡆⡆⠰⠀⠀⠀⠀⠛⢲⡴⠐⠆⡇⠀⠀⠀⠀⠀⠘⠀⢰⠀⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠇⢠⠨⠭⠿⣧⣤⠀⠀⠀⠀⠀⠨⠼⠷⣄⣧⠀⠀⠀⠀⠀⠀⠀⠸⠀⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢶⡆⡀⡀⠈⠐⠂⠀⠛⠻⣦⣀⡀⠀⠀⠀⠀⠘⠛⠛⠀⠀⠀⠀⠀⠀⠀⠀⠀⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢐⠚⢧⡇⢀⠀⠀⠀⠀⠀⠸⣿⣿⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⡆⠀⠹⢿⣤⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⡄⠀⠀⠀⠀⠀⢠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠁⠀⡆⠘⢘⡶⣂⡆⠀⠀⠀⠀⠀⠈⢻⣶⣦⠀⡇⠀⠀⠀⠀⠀⠘⠀⢰⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠃⢰⠰⠄⠙⢧⣶⠀⠀⠀⠀⠀⠈⠛⠻⣦⣷⠀⠀⠀⠀⠀⠀⠀⠘⠀⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠛⠀⠀⠀⠀⠈⠉⠉⠉⠛⠛⠀⠀⠀⠀⠉⠉⠉⠙⠛⠛⠀⠀⠀⠀⠀⠈⠀⠈⠀⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠒⠀⠀⠀⠀⠐⠒⠀⠀⠀⠀⠀⠀⠀⠀⠒⠒⠀⠀⠀⠂⠀⠀⠀⠀⠀⠐⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠒⠀⠀⠀⠀⠀⠐⠒⠒⠂⠀⠀⠀⠀⠀⠀⠐⠒⠒⠀⠂⠀⠀⠀⠀⠀⠀⠀⠐⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠉⠀⠀⠀⠀⠀⠈⠁⠀⠉⠉⠀⠀⠀⠀⠀⠀⠀⠈⠉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [31]:
fu = ∫(U ⋅ [0,0,0], Γ="remote")
fφ = ∫(Φ ⋅ [0,0,1e6], Γ="remote")

f = SystemVector([fu, fφ])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 281269, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 281269, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, not

In [32]:
u0, φ = solveField(Symmetric(K), f, support=[bc, bcz], mpc=[mpc_U, mpc_Φ], solver=:cholesky)

(VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 18.09709861383747; -0.009088040588974373;;], [0.0], Int64[], 1, :v3D, Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 281269, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing))), VectorField(Matrix{Float64}[], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0], Int64[], 1, :v3D, Problem("I-beam", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0), Material("remote", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 1666

In [33]:
u = u0 + R * φ

nodal VectorField
[0.0; 0.0; … ; 18.09709861383747; -0.009088040588974373;;]

In [34]:
showDoFResults(u, name="u", factor=1)

0

In [40]:
showDoFResults(φ, name="φ", factor=1e-5)

2

In [35]:
u = nodesToElements(u, onPhysicalGroup="beam")
ε = (u ∘ ∇ + ∇ ∘ u) / 2

E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [36]:
n = normalVector(U, "mid")

σ = elementsToElements(σ, fromPhysicalGroup="leftbeam", onPhysicalGroup="mid")

τ = σ * n
τ -= (τ ⋅ n) * n

τ = elementsToNodes(τ)
τ = nodesToElements(τ, onPhysicalGroup="mid")

elementwise VectorField
[[181.5386936148048; -182.87084561177454; … ; -245.31291154191211; 0.0;;], [36.61086788553556; -245.31291154191211; … ; -245.2385704372681; 0.0;;], [-38.80044046844313; -245.2385704372681; … ; -183.0292871991631; 0.0;;], [262.6018890119016; -51.92576638958822; … ; -80.52156590362037; 0.0;;], [67.47649461334488; -80.52156590362037; … ; -80.45477480365022; 0.0;;], [-72.34276434130173; -80.45477480365022; … ; -51.78977606943654; 0.0;;], [367.26604655850775; -17.985991844391084; … ; -26.44438737927696; 0.0;;], [110.4607854263268; -26.44438737927696; … ; -26.437786316472817; 0.0;;], [-119.94131535388138; -26.437786316472817; … ; -17.586413431719272; 0.0;;], [397.98117160082353; -6.15048618539073; … ; -8.716949805300676; 0.0;;]  …  [-128.88667854198016; 2.906390030329717; … ; 2.4647232594102606; 0.0;;], [411.9930337391818; 5.427531930399456; … ; 8.716682901549637; 0.0;;], [139.2352677574939; 8.716682901549637; … ; 8.716949805344177; 0.0;;], [-125.45624003007383; 8.716

In [37]:
r = nodePositionVector(U)
r = nodesToElements(r, onPhysicalGroup="mid")
∫(U, "mid", (r × τ) ⋅ n)

1.0721811537174555e6

In [38]:
showElementResults(τ, name="shear stress");

In [41]:
openPostProcessor()